## Импорты

In [1]:
import ipdb # <- трасировка и точки останова
import header
from header import __root__

# Internal modules
from src import gs

🔑 Found password in password.txt (DEBUG MODE)
✅ Successfully opened KeePass database: C:\Users\user\Documents\repos\hypotez\secrets\credentials.kdbx
Failed to load GAPI credentials


In [2]:
import re
import importlib
import os
import asyncio
import time
from pathlib import Path
from types import SimpleNamespace
from typing import Optional, List, Any
from dataclasses import dataclass, field


from src.suppliers.suppliers_list import *
from src.suppliers.get_graber_by_supplier  import get_graber_by_supplier_prefix, get_graber_by_supplier_url
from src.suppliers.graber import Graber
from src.webdriver.driver import Driver
from src.webdriver.firefox import Firefox
from src.webdriver.chrome import Chrome
from src.llm.gemini import GoogleGenerativeAi
from src.llm.openai.model import OpenAIModel
from src.endpoints.prestashop.product import PrestaProduct
from src.endpoints.prestashop.language import PrestaLanguage
from src.endpoints.prestashop.product_fields import ProductFields
from src.endpoints.advertisement.facebook.scenarios.post_message import (
    post_message,
)
from src.utils.file import read_text_file, save_text_file, get_filenames_from_directory

from src.utils.jjson import j_loads, j_loads_ns, j_dumps
from src.utils.image import get_image_bytes, get_raw_image_data
from src.utils.printer import pprint as print
from src.logger.logger import logger
from src.utils.file import recursively_get_file_path

2025-06-03 14:15:48,415 - INFO - Anonymized telemetry enabled. See https://docs.browser-use.com/development/telemetry for more information.


## Конфигурация

In [4]:
# --- file config.py
class Config:
    ENDPOINT: Path = __root__ /'SANDBOX' / 'davidka'
    SUPPLIERS_ENDPOINT: Path = __root__ / 'src' / 'suppliers' / 'suppliers_list'
    SCENARIOS_DIR:Path = __root__ /'SANDBOX' / 'davidka' / 'scenarios'
    config:SimpleNamespace = j_loads_ns(ENDPOINT / 'davidka.json')
    GEMINI_API_KEY:str = gs.credentials.gemini.onela.api_key
    PRESTA_API_KEY:str = gs.credentials.prestashop.store_davidka_net.api_key
    PRESTA_DOMAIN:str = gs.credentials.prestashop.store_davidka_net.api_domain
    gemini_model_name:str = config.gemini_model_name
    system_instruction:str = ' ' # <- Это пробел!
    webdriver_window_mode:str = 'headless'
# --- end file config.pt

In [5]:
# Config.PRESTA_API_KEY = 'IIYP1XSGRJESF9Z7B4T5YK47BSXZSKZB'
# Config.PRESTA_DOMAIN = 'https://store.davidka.net'
# p = PrestaProduct( Config.PRESTA_API_KEY, Config.PRESTA_DOMAIN )
# print(Config.PRESTA_API_KEY)
# print(Config.PRESTA_DOMAIN)

## Подключение драйвера

In [6]:
driver:Driver = None

try:
    driver = Driver(Firefox, window_mode = 'normal')
except Exception as ex:
    logger.critical(f'Ошибка инициализации шебдрайвера: ', ех, False)  

2025-06-03 14:16:44,626 - INFO - ℹ️ Инициализация Firefox WebDriver 
2025-06-03 14:16:44,627 - DEBUG - 🐛 Текущий __root__: C:\Users\user\Documents\repos\hypotez 
NoneType: None
2025-06-03 14:16:44,629 - DEBUG - 🐛 Конфигурация загружена. enable_geckodriver_log: True 
NoneType: None
2025-06-03 14:16:44,630 - INFO - ℹ️ Попытка настроить логирование geckodriver... 
2025-06-03 14:16:44,630 - DEBUG - 🐛 Предполагаемый путь к лог-файлу geckodriver: C:\Users\user\Documents\repos\hypotez\geckodriver.log 
NoneType: None
2025-06-03 14:16:44,633 - INFO - ℹ️ Логирование geckodriver настроено. Путь к лог-файлу: C:\Users\user\Documents\repos\hypotez\geckodriver.log 
2025-06-03 14:16:50,058 - INFO - ℹ️ Браузер Firefox успешно запущен. Режим окна: normal. 


In [7]:
async def execute_scenario(supplier_prefix:str, scenario:dict, driver:Driver):
    """"""
    ...
    supplier_alias = supplier_prefix.replace('.','_').replace('-','_')
    if not 'url' in scenario:
        logger.debug('Возможно новый поставщик у которго еще нет сценария категориий')
        return

    
    graber: Graber = None
    try:
        supplier_path:Path = Config.SUPPLIERS_ENDPOINT / supplier_prefix 
        graber = get_graber_by_supplier_prefix(supplier_prefix)
        scenarios_dict: dict = j_loads(Config.SCENARIOS_DIR  / f'{supplier_prefix}.json')
        locators_path:Path = supplier_path / 'locators' 
        locator_product:SimpleNamespace = j_loads_ns(locators_path / 'product.json')
        locator_category:SimpleNamespace = j_loads_ns(locators_path / 'category.json')
        categories_crawler:Any = None
        
    except Exception as ex:
        logger.error(f'Непредвиденная ошибка', ex)
        ...
        return False

    driver.get_url(scenario['url'])

    products_urls_list:list = await driver.execute_locator(locator_category.product_links) 

    required_fields:tuple = (
                            'name',
                            'id_supplier',
                            'description_short',
                            'description',
                            'specification',
                            'local_image_path',                      
                            'default_image_url',
                            'price'
                            )


    for url in products_urls_list:
        
        f:ProductFields = await graber.grab_page_async(*required_fields, url=url)
        ipdb.set_trace()
        ...
        


   
        ...

In [ ]:



suppliers_prefixes_list:list = ['aliexpress']  
products_urls_in_category:list = [] 
product_fields:ProductFields = None

#scenario: Scenario = Scenario(driver = driver)
# await scenario.process_suppliers_list(suppliers_prefixes_list)

    

## Сбор сценариев

In [8]:
scenario_files:list[Path] = recursively_get_file_path(Config.SCENARIOS_DIR)

def run_scenarios():
    for scenario_file in scenario_files:
        scenarios_dict:list[dict]|dict = j_loads(scenario_file)
        if not scenarios_dict: # в случае ошибки чтения файла json
            continue
    
        if isinstance(scenarios_dict, dict):
            ipdb.set_trace()
            execute_scenario(scenario_file.replace('.json',''), scenarios_dict, driver)
            
        elif isinstance(scenarios_dict, list):
            for scenario in scenarios_dict:
                ipdb.set_trace()
                execute_scenario(scenario_file.replace('.json',''), scenario, driver)
                
            ...
print(scenario_files)            

C:\Users\user\Documents\repos\hypotez\SANDBOX\davidka\scenarios\ads-tec-iit.com.json
C:\Users\user\Documents\repos\hypotez\SANDBOX\davidka\scenarios\aliexpress.json
C:\Users\user\Documents\repos\hypotez\SANDBOX\davidka\scenarios\amazon.json
C:\Users\user\Documents\repos\hypotez\SANDBOX\davidka\scenarios\apple.com.json
C:\Users\user\Documents\repos\hypotez\SANDBOX\davidka\scenarios\atlascopco.com.json
C:\Users\user\Documents\repos\hypotez\SANDBOX\davidka\scenarios\bangood.json
C:\Users\user\Documents\repos\hypotez\SANDBOX\davidka\scenarios\bucketmaster.com.cn.json
C:\Users\user\Documents\repos\hypotez\SANDBOX\davidka\scenarios\cdata.json
C:\Users\user\Documents\repos\hypotez\SANDBOX\davidka\scenarios\cisco.com.json
C:\Users\user\Documents\repos\hypotez\SANDBOX\davidka\scenarios\de-de.ring.com.json
C:\Users\user\Documents\repos\hypotez\SANDBOX\davidka\scenarios\de.hexcel.com.json
C:\Users\user\Documents\repos\hypotez\SANDBOX\davidka\scenarios\de.rs-online.com.json
C:\Users\user\Documents

In [17]:
scenario_file:str = fr"C:\Users\user\Documents\repos\hypotez\SANDBOX\davidka\scenarios\amazon.json"
scenarios_ns = j_loads_ns(scenario_file)

2025-06-03 14:25:36,707 - ERROR - ❌ JSON parsing error for string (first 100 chars): "C:\Users\user\Documents\repos\hypotez\SANDBOX\davidka\scenarios\amazon.json..." Expecting value: line 1 column 1 (char 0)
2025-06-03 14:25:36,708 - DEBUG - 🐛 Trying repair_json 
Traceback (most recent call last):
  File "C:\Users\user\Documents\repos\hypotez\src\utils\jjson.py", line 387, in _string_to_dict
    result:dict = json.loads(cleaned_string)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\anaconda3\Lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\anaconda3\Lib\json\decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\anaconda3\Lib\json\decoder.py", line 355, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 

In [9]:
print(scenarios_dict)

NameError: name 'scenarios_dict' is not defined

In [ ]:
graber = get_graber_by_supplier_prefix(driver, 'amazon')
print(graber.product_locator.price)

In [ ]:
# Не все поля товара надо заполнять. Вот кортеж необходимых полей:
required_fields:tuple = (
                        'name',
                        'id_supplier', # <- этот ID я передаю в локаторе товара в поле `id_supplier` прописываю id из прастасшоп
                        'price',
                        'description_short',
                        'description',
                        'specification',
                        'local_image_path',
                        'default_image_url')
f:ProductFields = await graber.grab_page_async(*required_fields)

In [ ]:
#print(f)

In [ ]:
f.id_supplier = 2800
f.id_category_default = 11247
#f.price = await driver.execute_locator(price_locator)
f.link_rewrite = gs.now
#f.description_short = f.description_short[:500]

In [ ]:
p.add_new_product(f)